In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Cross-City Crash Hotspot Detection\n",
    "\n",
    "**Status:** scaffold. Analysis cells are not yet filled in — they\n",
    "require the curated Parquet output from Member B's transformation\n",
    "layer, which is not yet built. Each section states what the analysis\n",
    "*will* show once the curated layer exists.\n",
    "\n",
    "## Purpose\n",
    "\n",
    "Detect crash hotspots by clustering latitude/longitude pairs with a\n",
    "density-based method (DBSCAN or HDBSCAN), and compare hotspot\n",
    "characteristics across the three sources. The identical clustering\n",
    "code runs on every city — the only thing that changes per source is\n",
    "the input partition, not the algorithm or its parameters.\n",
    "\n",
    "## Two rules for this notebook\n",
    "\n",
    "**Rule 1 — only cluster rows with valid coordinates.** Filter on\n",
    "`has_valid_coordinates = TRUE`, never on `latitude IS NOT NULL AND\n",
    "longitude IS NOT NULL`. The latter passes the sentinel `(0,0)` rows,\n",
    "which would put crashes in the Gulf of Guinea and produce a cluster\n",
    "there. This is the exact failure mode documented in\n",
    "`docs/data_dictionary.md`.\n",
    "\n",
    "**Rule 2 — compare hotspot *patterns*, not crash *totals*.** Reporting\n",
    "thresholds and coverage differ by city. Chicago counts crashes that\n",
    "meet one reporting threshold; the UK, a national dataset with its own\n",
    "police-force practices; NYC, its own standard. The pipeline exists to\n",
    "compare the *shape* of patterns (where hotspots are, when they occur,\n",
    "what severity mix they have), not raw counts. State this explicitly\n",
    "in the writeup and in any chart caption that touches cross-city totals.\n",
    "\n",
    "## Method choice: DBSCAN / HDBSCAN\n",
    "\n",
    "Why density-based clustering rather than k-means:\n",
    "\n",
    "- **No fixed k.** Crash hotspots are not evenly distributed; k-means\n",
    "  forces every crash into one of k clusters, including isolated crashes\n",
    "  that are not part of any hotspot. DBSCAN/HDBSCAN let isolated crashes\n",
    "  be labelled as noise.\n",
    "- **Non-spherical clusters.** Road networks produce elongated hotspots\n",
    "  (along arterials). k-means assumes spherical clusters and would split\n",
    "  those.\n",
    "- **Density varies by city.** Chicago's urban core is denser than outer\n",
    "  boroughs; HDBSCAN adapts to varying density, DBSCAN with a single\n",
    "  `eps` does not. Start with HDBSCAN if available; fall back to DBSCAN\n",
    "  with per-city `eps` tuning if HDBSCAN is not installed.\n",
    "\n",
    "## Parameters\n",
    "\n",
    "Clustering parameters are **not** universal across cities. `eps` in\n",
    "DBSCAN is in degrees of latitude/longitude at the equator; converting\n",
    "to meters requires a per-latitude scaling that differs for Chicago\n",
    "(~41.8°N), NYC (~40.7°N), and the UK (~51–58°N). The notebook either\n",
    "(a) projects coordinates to a local metric CRS before clustering, or\n",
    "(b) tunes `eps` per city and documents the value. HDBSCAN sidesteps\n",
    "the `eps` question but still has `min_cluster_size` and `min_samples`,\n",
    "which need to be chosen per city's density."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 0. Setup\n",
    "\n",
    "Imports, plotting, and a per-city loader that reads one partition of\n",
    "curated Parquet."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from pathlib import Path\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "# hdbscan is optional; sklearn's DBSCAN is the fallback.\n",
    "try:\n",
    "    import hdbscan\n",
    "    HAS_HDBSCAN = True\n",
    "except ImportError:\n",
    "    HAS_HDBSCAN = False\n",
    "\n",
    "from sklearn.cluster import DBSCAN\n",
    "\n",
    "sns.set_theme(style=\"whitegrid\")\n",
    "plt.rcParams[\"figure.dpi\"] = 110\n",
    "\n",
    "CURATED_ROOT = Path(\"../data/curated\")\n",
    "SOURCES = [\"chicago_us\", \"nyc_us\", \"uk_stats19\"]\n",
    "\n",
    "print(f\"HDBSCAN available: {HAS_HDBSCAN}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def load_spatial(source_id: str, year: int | None = None) -> pd.DataFrame:\n",
    "    \"\"\"Load only rows with valid coordinates, for one source.\n",
    "\n",
    "    Filters on has_valid_coordinates = TRUE (NOT on latitude IS NOT NULL),\n",
    "    so the (0,0) sentinel rows are excluded. See notebook header, Rule 1.\n",
    "    \"\"\"\n",
    "    base = CURATED_ROOT / f\"source_id={source_id}\"\n",
    "    if year is not None:\n",
    "        base = base / f\"year={year}\"\n",
    "    if not base.exists():\n",
    "        raise FileNotFoundError(\n",
    "            f\"Curated partition not found: {base}. Member B's transformation \"\n",
    "            f\"layer must be run first; see TASKS.md.\"\n",
    "        )\n",
    "    df = pd.read_parquet(\n",
    "        base,\n",
    "        columns=[\"crash_id\", \"source_id\", \"crash_timestamp_utc\",\n",
    "                 \"latitude\", \"longitude\", \"severity\",\n",
    "                 \"has_valid_coordinates\"],\n",
    "    )\n",
    "    return df[df[\"has_valid_coordinates\"]].copy()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Per-source coverage\n",
    "\n",
    "How many coordinate-bearing crashes per source. A sanity check before\n",
    "clustering — if a source has only a few hundred valid points, its\n",
    "clustering result will not be meaningful and should be labelled as\n",
    "such.\n",
    "\n",
    "Expected order-of-magnitude from profiling: Chicago ~1.09M valid,\n",
    "UK ~514K valid, NYC — fewer rows total but a similar proportion of\n",
    "valid coordinates once the 2.3% sentinel share is excluded."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# for s in SOURCES:\n",
    "#     df = load_spatial(s)\n",
    "#     print(f\"{s}: {len(df):,} coordinate-bearing crashes\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. The clustering function — one implementation, all cities\n",
    "\n",
    "This is the load-bearing cell: a single function that runs on any\n",
    "source's spatial partition. The point of the pipeline is that the\n",
    "same clustering code works on every city, so this function must not\n",
    "branch on `source_id`. Any per-city differences belong in the\n",
    "parameters passed in, not in `if source_id == ...` inside the function.\n",
    "\n",
    "Returns the input DataFrame plus a `cluster_id` column (`-1` for noise,\n",
    "per DBSCAN/HDBSCAN convention)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def cluster_hotspots(\n",
    "    df: pd.DataFrame,\n",
    "    method: str = \"hdbscan\",\n",
    "    min_cluster_size: int = 30,\n",
    "    eps_degrees: float = 0.0015,\n",
    "    min_samples: int = 10,\n",
    ") -> pd.DataFrame:\n",
    "    \"\"\"Cluster crash coordinates into hotspots.\n",
    "\n",
    "    Parameters\n",
    "    ----------\n",
    "    df : DataFrame with 'latitude' and 'longitude' columns.\n",
    "    method : 'hdbscan' (default if installed) or 'dbscan'.\n",
    "    min_cluster_size : HDBSCAN parameter; minimum points to call a cluster.\n",
    "    eps_degrees : DBSCAN parameter; ~0.0015 deg is ~150 m at UK latitudes.\n",
    "                  NOTE: this is latitude-dependent. Tune per city or\n",
    "                  project to a metric CRS before relying on it.\n",
    "    min_samples : DBSCAN / HDBSCAN neighborhood size.\n",
    "\n",
    "    Returns\n",
    "    -------\n",
    "    Copy of df with an added 'cluster_id' column. -1 means noise.\n",
    "    \"\"\"\n",
    "    coords = df[[\"latitude\", \"longitude\"]].to_numpy()\n",
    "\n",
    "    if method == \"hdbscan\" and HAS_HDBSCAN:\n",
    "        clusterer = hdbscan.HDBSCAN(\n",
    "            min_cluster_size=min_cluster_size,\n",
    "            min_samples=min_samples,\n",
    "            metric=\"haversine\",\n",
    "        )\n",
    "        labels = clusterer.fit_predict(np.radians(coords))\n",
    "    elif method == \"dbscan\" or not HAS_HDBSCAN:\n",
    "        clusterer = DBSCAN(\n",
    "            eps=eps_degrees,\n",
    "            min_samples=min_samples,\n",
    "            metric=\"haversine\",\n",
    "        )\n",
    "        labels = clusterer.fit_predict(np.radians(coords))\n",
    "    else:\n",
    "        raise ValueError(f\"Unknown method: {method}\")\n",
    "\n",
    "    out = df.copy()\n",
    "    out[\"cluster_id\"] = labels\n",
    "    return out"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Cluster one source (example: Chicago)\n",
    "\n",
    "Run the same function on one source end-to-end so the full pipeline\n",
    "(load → cluster → summarize) is visible. Later sections loop over\n",
    "all three sources.\n",
    "\n",
    "Parameters shown here (`min_cluster_size`, `min_samples`) are starting\n",
    "points. Real values should be tuned per city — see header note."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# df_chi = load_spatial(\"chicago_us\")\n",
    "# df_chi = cluster_hotspots(df_chi, method=\"hdbscan\", min_cluster_size=30, min_samples=10)\n",
    "# n_clusters = df_chi.loc[df_chi[\"cluster_id\"] >= 0, \"cluster_id\"].nunique()\n",
    "# n_noise = (df_chi[\"cluster_id\"] == -1).sum()\n",
    "# print(f\"Chicago: {n_clusters} clusters, {n_noise:,} noise points \"\n",
    "#       f\"({n_noise/len(df_chi):.1%} of rows)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Summarize a clustering result\n",
    "\n",
    "For each cluster: size, centroid, severity mix, temporal profile.\n",
    "This is the shape that goes into the writeup — a table of hotspots\n",
    "with their characteristics, not just a scatter plot.\n",
    "\n",
    "The same summarization runs on any source, so the outputs are directly\n",
    "comparable in structure."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def summarize_clusters(df: pd.DataFrame) -> pd.DataFrame:\n",
    "    \"\"\"Per-cluster summary: size, centroid, severity mix, hour profile.\n",
    "\n",
    "    Excludes cluster_id == -1 (noise).\n",
    "    \"\"\"\n",
    "    clustered = df[df[\"cluster_id\"] >= 0]\n",
    "    if clustered.empty:\n",
    "        return pd.DataFrame(\n",
    "            columns=[\"cluster_id\", \"size\", \"centroid_lat\", \"centroid_lon\",\n",
    "                     \"pct_fatal\", \"pct_serious\", \"pct_minor\"]\n",
    "        )\n",
    "\n",
    "    def sev_share(g, level):\n",
    "        return (g[\"severity\"] == level).mean()\n",
    "\n",
    "    grouped = clustered.groupby(\"cluster_id\")\n",
    "    return pd.DataFrame({\n",
    "        \"size\": grouped.size(),\n",
    "        \"centroid_lat\": grouped[\"latitude\"].mean(),\n",
    "        \"centroid_lon\": grouped[\"longitude\"].mean(),\n",
    "        \"pct_fatal\": grouped.apply(lambda g: sev_share(g, \"fatal\")),\n",
    "        \"pct_serious\": grouped.apply(lambda g: sev_share(g, \"serious\")),\n",
    "        \"pct_minor\": grouped.apply(lambda g: sev_share(g, \"minor\")),\n",
    "    }).sort_values(\"size\", ascending=False)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Run clustering across all three sources\n",
    "\n",
    "The loop that produces the cross-city comparison. Same function, same\n",
    "summarization, three inputs.\n",
    "\n",
    "**Structural gap to annotate:** the `severity` mix within a cluster is\n",
    "not comparable across all three sources. NYC cannot produce\n",
    "`serious`, so its cluster summaries will show 0% serious by\n",
    "construction, not by finding. When comparing severity mix across\n",
    "cities, restrict to Chicago + UK, or collapse `serious` and `minor`\n",
    "into a single `injury` bucket for the NYC-inclusive comparison. State\n",
    "which choice was made in the writeup."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# results = {}\n",
    "# for s in SOURCES:\n",
    "#     df_s = load_spatial(s)\n",
    "#     df_s = cluster_hotspots(df_s, method=\"hdbscan\",\n",
    "#                             min_cluster_size=30, min_samples=10)\n",
    "#     results[s] = summarize_clusters(df_s)\n",
    "#     print(f\"{s}: {len(results[s])} clusters\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Visualize hotspots — one city at a time\n",
    "\n",
    "A scatter of crash points colored by cluster ID, per city, on its own\n",
    "axes. Do **not** overlay all three cities on one map: they occupy\n",
    "different geographies and different coordinate ranges, and overlaying\n",
    "them makes each one's structure unreadable."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def plot_hotspots(df: pd.DataFrame, title: str):\n",
    "    \"\"\"Scatter crashes colored by cluster_id. Noise (-1) is grey.\"\"\"\n",
    "    fig, ax = plt.subplots(figsize=(8, 8))\n",
    "    clustered = df[df[\"cluster_id\"] >= 0]\n",
    "    noise = df[df[\"cluster_id\"] == -1]\n",
    "    ax.scatter(noise[\"longitude\"], noise[\"latitude\"],\n",
    "               s=1, c=\"lightgrey\", label=\"noise\", alpha=0.5)\n",
    "    ax.scatter(clustered[\"longitude\"], clustered[\"latitude\"],\n",
    "               s=2, c=clustered[\"cluster_id\"], cmap=\"tab20\", alpha=0.7)\n",
    "    ax.set_title(title)\n",
    "    ax.set_xlabel(\"longitude\")\n",
    "    ax.set_ylabel(\"latitude\")\n",
    "    ax.set_aspect(\"equal\", adjustable=\"datalim\")\n",
    "    plt.tight_layout()\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Cluster stability\n",
    "\n",
    "Cross-city comparison is only meaningful if the clusters are stable —\n",
    "re-running with slightly different parameters should produce roughly\n",
    "the same hotspot geography, not a completely different one.\n",
    "\n",
    "A simple check: re-run clustering on a random 80% subsample of each\n",
    "city's points and compare the resulting cluster count and mean cluster\n",
    "size. Large swings indicate the parameters are too sensitive for the\n",
    "result to be interpreted."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# for s in SOURCES:\n",
    "#     df_s = load_spatial(s)\n",
    "#     for seed in [0, 1, 2]:\n",
    "#         sample = df_s.sample(frac=0.8, random_state=seed)\n",
    "#         clustered = cluster_hotspots(sample)\n",
    "#         n_clusters = clustered.loc[clustered[\"cluster_id\"] >= 0, \"cluster_id\"].nunique()\n",
    "#         print(f\"{s} seed={seed}: {n_clusters} clusters\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Notes for whoever fills this in\n",
    "\n",
    "1. **Filter on `has_valid_coordinates`, not on coordinate non-nullness.**\n",
    "   The sentinel `(0,0)` rows will pass a null check and place crashes\n",
    "   in the Gulf of Guinea. This is the exact failure mode the pipeline's\n",
    "   `has_valid_coordinates` column exists to prevent.\n",
    "\n",
    "2. **Do not branch on `source_id` inside the clustering function.**\n",
    "   The whole point of the pipeline is that the same algorithm applies\n",
    "   to every city. Per-city differences belong in parameters\n",
    "   (`min_cluster_size`, `eps`), passed in by the caller.\n",
    "\n",
    "3. **Choose `eps` (DBSCAN) with latitude in mind.** `eps` is in\n",
    "   degrees. The same numeric `eps` means a different physical distance\n",
    "   at Chicago's latitude (~41.8°N) vs the UK's (~55°N). Either project\n",
    "   to a metric CRS before clustering, or tune `eps` per city and\n",
    "   document the chosen values.\n",
    "\n",
    "4. **Report the noise fraction.** DBSCAN/HDBSCAN leave some points\n",
    "   unclustered. A noise fraction above ~30% is a signal that the\n",
    "   parameters are wrong for the data's density, not that the data has\n",
    "   no hotspots.\n",
    "\n",
    "5. **State the cross-city caveat in the writeup.** Crash counts depend\n",
    "   on each city's reporting thresholds and practices. The notebook's\n",
    "   contribution is that hotspot *patterns* are comparable — where\n",
    "   clusters are, what their severity mix looks like, when their\n",
    "   crashes occur — not that crash *totals* are comparable across\n",
    "   cities. Do not present a cross-city bar chart of raw crash counts\n",
    "   as a finding.\n",
    "\n",
    "6. **Document the parameters used, in the notebook, for each run.**\n",
    "   `min_cluster_size`, `min_samples`, `eps`, and the method chosen.\n",
    "   HDBSCAN's defaults are not always right; DBSCAN's `eps` is\n",
    "   almost never right without tuning. Whoever reads the results\n",
    "   needs to know what produced them."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}